Fetching Data

In [ ]:
"""
Study-Abroad Affordability Comparison Tool
Fully API-based — no manual data entry, no modeling.
Uses: World Bank Open Data API (free, no key) + REST Countries API (free, no key)
"""

import requests
import pandas as pd

# ============================================================
# STEP 1: Pick the countries to compare (edit this list anytime)
# ============================================================
# ISO-3 codes: Pakistan, USA, UK, Germany, Canada, Australia
countries = {
    "PAK": "Pakistan",
    "USA": "United States",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "CAN": "Canada",
    "AUS": "Australia",
}
country_codes = ";".join(countries.keys())

# ============================================================
# STEP 2: Indicators that approximate "cost of living / affordability"
# ============================================================
indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",     # purchasing power per person
    "PA.NUS.PPPC.RF": "price_level_ratio",          # how expensive vs. US dollar
    "FP.CPI.TOTL.ZG": "inflation_pct",              # cost rising how fast
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp", # rough tuition/education signal
}

BASE_URL = "https://api.worldbank.org/v2/country"

def fetch_indicator(indicator_code, countries_str, year="2023"):
    """Fetch one indicator for multiple countries from the World Bank API."""
    url = f"{BASE_URL}/{countries_str}/indicator/{indicator_code}"
    params = {"format": "json", "date": year, "per_page": 100}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if len(data) < 2 or data[1] is None:
        return pd.DataFrame(columns=["country_code", "country_name", "year", "value"])

    rows = []
    for rec in data[1]:
        rows.append({
            "country_code": rec["countryiso3code"],
            "country_name": rec["country"]["value"],
            "year": rec["date"],
            "value": rec["value"],
        })
    return pd.DataFrame(rows)

# ============================================================
# STEP 3: Pull every indicator and merge into one table
# ============================================================
merged = None
for code, name in indicators.items():
    df = fetch_indicator(code, country_codes)
    df = df.rename(columns={"value": name})[["country_code", "country_name", name]]
    merged = df if merged is None else merged.merge(df, on=["country_code", "country_name"], how="outer")

print("=== Raw data pulled from World Bank API ===")
print(merged.to_string(index=False))

# ============================================================
# STEP 4: Pull basic country info from REST Countries API (free, no key)
# ============================================================
def fetch_country_info(name):
    url = f"https://restcountries.com/v3.1/name/{name}"
    try:
        resp = requests.get(url, params={"fields": "name,currencies,region,population"}, timeout=15)
        resp.raise_for_status()
        d = resp.json()[0]
        currency_code = list(d.get("currencies", {}).keys())[0] if d.get("currencies") else None
        return {
            "region": d.get("region"),
            "currency": currency_code,
            "population": d.get("population"),
        }
    except Exception:
        return {"region": None, "currency": None, "population": None}

info_rows = []
for code, name in countries.items():
    info = fetch_country_info(name)
    info["country_code"] = code
    info_rows.append(info)

info_df = pd.DataFrame(info_rows)
merged = merged.merge(info_df, on="country_code", how="left")

# ============================================================
# STEP 5: Build a simple "affordability score" (no model, just plain math)
# ============================================================
# Lower price level + lower inflation + higher purchasing power = more affordable
merged["affordability_score"] = (
    merged["gni_per_capita_ppp"].rank(ascending=False)       # higher is better -> rank 1 = best
    + merged["price_level_ratio"].rank(ascending=True)        # lower is better
    + merged["inflation_pct"].rank(ascending=True)             # lower is better
)
merged = merged.sort_values("affordability_score")

print("\n=== Final comparison table (lower affordability_score = more affordable) ===")
print(merged.to_string(index=False))

merged.to_csv("study_abroad_comparison.csv", index=False)
print("\nSaved -> study_abroad_comparison.csv")

# ============================================================
# STEP 6: Simple lookup function — pick any 2 countries to compare
# ============================================================
def compare(country_a, country_b, df):
    sub = df[df["country_name"].isin([country_a, country_b])]
    if len(sub) < 2:
        print("One or both countries not found in the pulled data.")
        return
    print(f"\n--- {country_a} vs {country_b} ---")
    print(sub[["country_name", "gni_per_capita_ppp", "price_level_ratio",
               "inflation_pct", "affordability_score"]].to_string(index=False))

compare("Pakistan", "United Kingdom", merged)

=== Raw data pulled from World Bank API ===
country_code   country_name  gni_per_capita_ppp price_level_ratio  inflation_pct  education_spend_pct_gdp
         AUS      Australia               69080               NaN       5.597018                      NaN
         CAN         Canada               63950               NaN       3.879002                      NaN
         DEU        Germany               74600               NaN       5.946437                      NaN
         GBR United Kingdom               59770               NaN       6.793967                      NaN
         PAK       Pakistan                5920               NaN      30.768128                  1.94658
         USA  United States               81870               NaN       4.116338                      NaN

=== Final comparison table (lower affordability_score = more affordable) ===
country_code   country_name  gni_per_capita_ppp price_level_ratio  inflation_pct  education_spend_pct_gdp region currency population  af

the problem was in the API i was getting Nan values

In [ ]:
"""
Study-Abroad Affordability Comparison Tool
Fully API-based — no manual data entry, no modeling.
Uses: World Bank Open Data API (free, no key) + REST Countries API (free, no key)
"""

import requests
import pandas as pd
import numpy as np

# ============================================================
# STEP 1: Pick the countries to compare (edit this list anytime)
# ============================================================
countries = {
    "PAK": "Pakistan",
    "USA": "United States",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "CAN": "Canada",
    "AUS": "Australia",
}
country_codes = ";".join(countries.keys())

# ============================================================
# STEP 2: Indicators that approximate "cost of living / affordability"
# ============================================================
indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",
    "PA.NUS.PPPC.RF": "price_level_ratio",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp",
}

BASE_URL = "https://api.worldbank.org/v2/country"


def fetch_indicator(indicator_code, countries_str, year_range="2010:2023"):
    """Fetch one indicator for multiple countries, keep the most recent
       non-missing value per country. Prints what the API actually returned,
       so a bad indicator code or empty response is visible immediately."""
    url = f"{BASE_URL}/{countries_str}/indicator/{indicator_code}"
    params = {"format": "json", "date": year_range, "per_page": 1000}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    print(f"  -> API response for {indicator_code}: {data[0]}")  # shows total/pages/message

    if len(data) < 2 or data[1] is None:
        return pd.DataFrame(columns=["country_code", "country_name", "value", "year"])

    rows = []
    for rec in data[1]:
        if rec["value"] is not None:
            rows.append({
                "country_code": rec["countryiso3code"],
                "country_name": rec["country"]["value"],
                "year": int(rec["date"]),
                "value": rec["value"],
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["country_code", "country_name", "value", "year"])
    df = df.sort_values("year").groupby("country_code", as_index=False).tail(1)
    return df


# ============================================================
# STEP 3: Pull every indicator and merge into one table
#          (always creates the column, even if the API gave nothing)
# ============================================================
base = pd.DataFrame({
    "country_code": list(countries.keys()),
    "country_name": list(countries.values()),
})

merged = base.copy()
for code, name in indicators.items():
    print(f"Fetching {name} ({code})...")
    df = fetch_indicator(code, country_codes)
    if df.empty:
        print(f"  WARNING: no data returned for {name} ({code}) — filling with NaN\n")
        merged[name] = np.nan
        merged[f"{name}_year"] = np.nan
        continue
    df = df.rename(columns={"value": name, "year": f"{name}_year"})
    df = df[["country_code", name, f"{name}_year"]]
    merged = merged.merge(df, on="country_code", how="left")

print("=== Raw data pulled from World Bank API ===")
print(merged.to_string(index=False))

# ============================================================
# STEP 4: Pull basic country info from REST Countries API (free, no key)
# ============================================================
def fetch_country_info(name):
    url = f"https://restcountries.com/v3.1/name/{name}"
    try:
        resp = requests.get(url, params={"fields": "name,currencies,region,population"}, timeout=15)
        resp.raise_for_status()
        d = resp.json()[0]
        currency_code = list(d.get("currencies", {}).keys())[0] if d.get("currencies") else None
        return {"region": d.get("region"), "currency": currency_code, "population": d.get("population")}
    except Exception:
        return {"region": None, "currency": None, "population": None}


info_rows = []
for code, name in countries.items():
    info = fetch_country_info(name)
    info["country_code"] = code
    info_rows.append(info)

info_df = pd.DataFrame(info_rows)
merged = merged.merge(info_df, on="country_code", how="left")

# ============================================================
# STEP 5: Build a simple "affordability score" (no model, just plain math)
#          Skips any indicator that's entirely missing, instead of crashing.
# ============================================================
score_parts = []

if merged["gni_per_capita_ppp"].notna().any():
    score_parts.append(merged["gni_per_capita_ppp"].rank(ascending=False))
if merged["price_level_ratio"].notna().any():
    score_parts.append(merged["price_level_ratio"].rank(ascending=True))
if merged["inflation_pct"].notna().any():
    score_parts.append(merged["inflation_pct"].rank(ascending=True))

if score_parts:
    merged["affordability_score"] = sum(score_parts)
    merged = merged.sort_values("affordability_score")
else:
    merged["affordability_score"] = np.nan
    print("\nWARNING: no usable indicators to score with.")

print("\n=== Final comparison table (lower affordability_score = more affordable) ===")
print(merged.to_string(index=False))

merged.to_csv("study_abroad_comparison.csv", index=False)
print("\nSaved -> study_abroad_comparison.csv")

# ============================================================
# STEP 6: Simple lookup function — pick any 2 countries to compare
# ============================================================
def compare(country_a, country_b, df):
    sub = df[df["country_name"].isin([country_a, country_b])]
    if len(sub) < 2:
        print("One or both countries not found in the pulled data.")
        return
    cols = ["country_name"] + [c for c in
            ["gni_per_capita_ppp", "price_level_ratio", "inflation_pct", "affordability_score"]
            if c in df.columns]
    print(f"\n--- {country_a} vs {country_b} ---")
    print(sub[cols].to_string(index=False))


compare("Pakistan", "United Kingdom", merged)

Fetching gni_per_capita_ppp (NY.GNP.PCAP.PP.CD)...
  -> API response for NY.GNP.PCAP.PP.CD: {'page': 1, 'pages': 1, 'per_page': 1000, 'total': 84, 'sourceid': '2', 'lastupdated': '2026-10-08'}
Fetching price_level_ratio (PA.NUS.PPPC.RF)...
  -> API response for PA.NUS.PPPC.RF: {'message': [{'id': '175', 'key': 'Invalid format', 'value': 'The indicator was not found. It may have been deleted or archived.'}]}

Fetching inflation_pct (FP.CPI.TOTL.ZG)...
  -> API response for FP.CPI.TOTL.ZG: {'page': 1, 'pages': 1, 'per_page': 1000, 'total': 84, 'sourceid': '2', 'lastupdated': '2026-10-08'}
Fetching education_spend_pct_gdp (SE.XPD.TOTL.GD.ZS)...
  -> API response for SE.XPD.TOTL.GD.ZS: {'page': 1, 'pages': 1, 'per_page': 1000, 'total': 84, 'sourceid': '2', 'lastupdated': '2026-10-08'}
=== Raw data pulled from World Bank API ===
country_code   country_name  gni_per_capita_ppp  gni_per_capita_ppp_year  price_level_ratio  price_level_ratio_year  inflation_pct  inflation_pct_year  education_sp

Here i resolved the issue of Nan values

In [ ]:
"""
Study-Abroad Affordability Comparison Tool
Fully API-based — no manual data entry, no modeling.
Uses: World Bank Open Data API (free, no key) + REST Countries API (free, no key)
"""

import requests
import pandas as pd
import numpy as np

# ============================================================
# STEP 1: Pick the countries to compare
# ============================================================
countries = {
    "PAK": "Pakistan",
    "USA": "United States",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "CAN": "Canada",
    "AUS": "Australia",
}
country_codes = ";".join(countries.keys())

# ============================================================
# STEP 2: Indicators (price_level_ratio built from 2 live indicators,
#          since PA.NUS.PPPC.RF is archived and no longer works)
# ============================================================
indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",
    "PA.NUS.PPP": "ppp_conversion_factor",
    "PA.NUS.FCRF": "official_exchange_rate",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp",
}

BASE_URL = "https://api.worldbank.org/v2/country"


def fetch_indicator(indicator_code, countries_str, year_range="2010:2023"):
    url = f"{BASE_URL}/{countries_str}/indicator/{indicator_code}"
    params = {"format": "json", "date": year_range, "per_page": 1000}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if len(data) < 2 or data[1] is None:
        print(f"  WARNING: {indicator_code} returned: {data[0]}")
        return pd.DataFrame(columns=["country_code", "value", "year"])

    rows = []
    for rec in data[1]:
        if rec["value"] is not None:
            rows.append({
                "country_code": rec["countryiso3code"],
                "year": int(rec["date"]),
                "value": rec["value"],
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values("year").groupby("country_code", as_index=False).tail(1)


# ============================================================
# STEP 3: Pull every indicator and merge
# ============================================================
base = pd.DataFrame({
    "country_code": list(countries.keys()),
    "country_name": list(countries.values()),
})

merged = base.copy()
for code, name in indicators.items():
    print(f"Fetching {name} ({code})...")
    df = fetch_indicator(code, country_codes)
    if df.empty:
        print(f"  -> no data for {name}, filling NaN\n")
        merged[name] = np.nan
        continue
    df = df.rename(columns={"value": name})[["country_code", name]]
    merged = merged.merge(df, on="country_code", how="left")

# Build price_level_ratio ourselves (same formula the old indicator used)
merged["price_level_ratio"] = merged["ppp_conversion_factor"] / merged["official_exchange_rate"]

print("=== Raw + calculated data ===")
print(merged.to_string(index=False))

# ============================================================
# STEP 4: REST Countries API — with visible error handling
# ============================================================
def fetch_country_info(name):
    url = f"https://restcountries.com/v3.1/name/{name}"
    try:
        resp = requests.get(url, params={"fields": "name,currencies,region,population"}, timeout=15)
        resp.raise_for_status()
        results = resp.json()
        # pick the exact match if multiple countries share part of the name
        d = next((r for r in results if r["name"]["common"].lower() == name.lower()), results[0])
        currency_code = list(d.get("currencies", {}).keys())[0] if d.get("currencies") else None
        return {"region": d.get("region"), "currency": currency_code, "population": d.get("population")}
    except Exception as e:
        print(f"  REST Countries error for {name}: {e}")
        return {"region": None, "currency": None, "population": None}


info_rows = []
for code, name in countries.items():
    info = fetch_country_info(name)
    info["country_code"] = code
    info_rows.append(info)

info_df = pd.DataFrame(info_rows)
merged = merged.merge(info_df, on="country_code", how="left")

# ============================================================
# STEP 5: Affordability score (plain math, no model)
# ============================================================
score_parts = []
if merged["gni_per_capita_ppp"].notna().any():
    score_parts.append(merged["gni_per_capita_ppp"].rank(ascending=False))
if merged["price_level_ratio"].notna().any():
    score_parts.append(merged["price_level_ratio"].rank(ascending=True))
if merged["inflation_pct"].notna().any():
    score_parts.append(merged["inflation_pct"].rank(ascending=True))

merged["affordability_score"] = sum(score_parts) if score_parts else np.nan
merged = merged.sort_values("affordability_score")

print("\n=== Final comparison table (lower affordability_score = more affordable) ===")
print(merged.to_string(index=False))

merged.to_csv("study_abroad_comparison.csv", index=False)
print("\nSaved -> study_abroad_comparison.csv")


# ============================================================
# STEP 6: Compare any two countries
# ============================================================
def compare(country_a, country_b, df):
    sub = df[df["country_name"].isin([country_a, country_b])]
    if len(sub) < 2:
        print("One or both countries not found.")
        return
    cols = ["country_name", "gni_per_capita_ppp", "price_level_ratio",
            "inflation_pct", "affordability_score"]
    print(f"\n--- {country_a} vs {country_b} ---")
    print(sub[cols].to_string(index=False))


compare("Pakistan", "United Kingdom", merged)

Fetching gni_per_capita_ppp (NY.GNP.PCAP.PP.CD)...
Fetching ppp_conversion_factor (PA.NUS.PPP)...
Fetching official_exchange_rate (PA.NUS.FCRF)...
Fetching inflation_pct (FP.CPI.TOTL.ZG)...
Fetching education_spend_pct_gdp (SE.XPD.TOTL.GD.ZS)...
=== Raw + calculated data ===
country_code   country_name  gni_per_capita_ppp  ppp_conversion_factor  official_exchange_rate  inflation_pct  education_spend_pct_gdp  price_level_ratio
         PAK       Pakistan                5920              56.152188              280.356112      30.768128                  1.94658           0.200289
         USA  United States               81870               1.000000                1.000000       4.116338                  5.42038           1.000000
         GBR United Kingdom               59770               0.660709                0.804539       6.793967                  5.90682           0.821227
         DEU        Germany               74600               0.701054                0.924840       5.94643

building the affordobality score without modeling pure math

In [ ]:
import requests

def fetch_country_info(name):
    url = f"https://restcountries.com/v3.1/name/{name}"
    try:
        resp = requests.get(url, params={"fields": "name,currencies,region,population"}, timeout=15)
        print(f"  [{name}] status={resp.status_code}  body preview: {resp.text[:200]}")
        resp.raise_for_status()
        results = resp.json()

        if isinstance(results, dict):
            results = [results]
        if not results:
            print(f"  [{name}] empty results")
            return {"region": None, "currency": None, "population": None}

        d = next((r for r in results if r.get("name", {}).get("common", "").lower() == name.lower()), results[0])
        currency_code = list(d.get("currencies", {}).keys())[0] if d.get("currencies") else None
        return {"region": d.get("region"), "currency": currency_code, "population": d.get("population")}

    except Exception as e:
        print(f"  REST Countries error for {name}: {type(e).__name__}: {e}")
        return {"region": None, "currency": None, "population": None}

# Actually call it, so something prints:
result = fetch_country_info("Pakistan")
print("\nFinal result:", result)

  [Pakistan] status=200  body preview: {
    "success": false,
    "data": null,
    "errors": [
        {
            "message": "This API version has been deprecated. Please visit https://restcountries.com/docs/countries/legacy-api-depre

Final result: {'region': None, 'currency': None, 'population': None}


In [ ]:
"""
Study-Abroad Affordability Comparison Tool
Fully API-based — no manual data entry, no modeling.
Uses: World Bank Open Data API only (free, no key).
(REST Countries API removed — its provider shut down the free version.)
"""

import requests
import pandas as pd
import numpy as np

# ============================================================
# STEP 1: Pick the countries to compare (edit this list anytime)
# ============================================================
countries = {
    "PAK": "Pakistan",
    "USA": "United States",
    "GBR": "United Kingdom",
    "DEU": "Germany",
    "CAN": "Canada",
    "AUS": "Australia",
}
country_codes = ";".join(countries.keys())

# ============================================================
# STEP 2: Indicators (price_level_ratio is built from 2 live
#          indicators, since the old precomputed one was archived)
# ============================================================
indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",
    "PA.NUS.PPP": "ppp_conversion_factor",
    "PA.NUS.FCRF": "official_exchange_rate",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp",
}

BASE_URL = "https://api.worldbank.org/v2/country"


def fetch_indicator(indicator_code, countries_str, year_range="2010:2023"):
    """Fetch one indicator for multiple countries, keep the most recent
       non-missing value per country."""
    url = f"{BASE_URL}/{countries_str}/indicator/{indicator_code}"
    params = {"format": "json", "date": year_range, "per_page": 1000}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if len(data) < 2 or data[1] is None:
        print(f"  WARNING: {indicator_code} returned: {data[0]}")
        return pd.DataFrame(columns=["country_code", "value", "year"])

    rows = []
    for rec in data[1]:
        if rec["value"] is not None:
            rows.append({
                "country_code": rec["countryiso3code"],
                "year": int(rec["date"]),
                "value": rec["value"],
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values("year").groupby("country_code", as_index=False).tail(1)


# ============================================================
# STEP 3: Pull every indicator and merge into one table
# ============================================================
base = pd.DataFrame({
    "country_code": list(countries.keys()),
    "country_name": list(countries.values()),
})

merged = base.copy()
for code, name in indicators.items():
    print(f"Fetching {name} ({code})...")
    df = fetch_indicator(code, country_codes)
    if df.empty:
        print(f"  -> no data for {name}, filling NaN\n")
        merged[name] = np.nan
        continue
    df = df.rename(columns={"value": name})[["country_code", name]]
    merged = merged.merge(df, on="country_code", how="left")

# Price level ratio = PPP conversion factor / official exchange rate
merged["price_level_ratio"] = merged["ppp_conversion_factor"] / merged["official_exchange_rate"]

print("=== Raw + calculated data ===")
print(merged.to_string(index=False))

# ============================================================
# STEP 4: Affordability score (plain math, no model)
#          Lower price level + lower inflation + higher purchasing
#          power = more affordable.
# ============================================================
score_parts = []
if merged["gni_per_capita_ppp"].notna().any():
    score_parts.append(merged["gni_per_capita_ppp"].rank(ascending=False))
if merged["price_level_ratio"].notna().any():
    score_parts.append(merged["price_level_ratio"].rank(ascending=True))
if merged["inflation_pct"].notna().any():
    score_parts.append(merged["inflation_pct"].rank(ascending=True))

merged["affordability_score"] = sum(score_parts) if score_parts else np.nan
merged = merged.sort_values("affordability_score")

print("\n=== Final comparison table (lower affordability_score = more affordable) ===")
print(merged.to_string(index=False))

merged.to_csv("study_abroad_comparison.csv", index=False)
print("\nSaved -> study_abroad_comparison.csv")


# ============================================================
# STEP 5: Compare any two countries — this is what your deployed
#          tool calls when a student picks two countries
# ============================================================
def compare(country_a, country_b, df):
    sub = df[df["country_name"].isin([country_a, country_b])]
    if len(sub) < 2:
        print("One or both countries not found.")
        return
    cols = ["country_name", "gni_per_capita_ppp", "price_level_ratio",
            "inflation_pct", "education_spend_pct_gdp", "affordability_score"]
    print(f"\n--- {country_a} vs {country_b} ---")
    print(sub[cols].to_string(index=False))


compare("Pakistan", "United Kingdom", merged)

Fetching gni_per_capita_ppp (NY.GNP.PCAP.PP.CD)...
Fetching ppp_conversion_factor (PA.NUS.PPP)...
Fetching official_exchange_rate (PA.NUS.FCRF)...
Fetching inflation_pct (FP.CPI.TOTL.ZG)...
Fetching education_spend_pct_gdp (SE.XPD.TOTL.GD.ZS)...
=== Raw + calculated data ===
country_code   country_name  gni_per_capita_ppp  ppp_conversion_factor  official_exchange_rate  inflation_pct  education_spend_pct_gdp  price_level_ratio
         PAK       Pakistan                5920              56.152188              280.356112      30.768128                  1.94658           0.200289
         USA  United States               81870               1.000000                1.000000       4.116338                  5.42038           1.000000
         GBR United Kingdom               59770               0.660709                0.804539       6.793967                  5.90682           0.821227
         DEU        Germany               74600               0.701054                0.924840       5.94643

i was getting only limited countries so i extracted all

In [ ]:
"""
Study-Abroad Affordability Comparison Tool — ALL countries, lighter version
"""

import requests
import pandas as pd
import numpy as np
import time

indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",
    "PA.NUS.PPP": "ppp_conversion_factor",
    "PA.NUS.FCRF": "official_exchange_rate",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp",
}

BASE_URL = "https://api.worldbank.org/v2/country"

def fetch_indicator_all(indicator_code, retries=3):
    """Fetch the most recent non-empty value per country (mrnev=1).
       Much lighter than pulling a full date range for all countries."""
    url = f"{BASE_URL}/all/indicator/{indicator_code}"
    params = {"format": "json", "mrnev": 1, "per_page": 400}

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            break
        except requests.exceptions.ReadTimeout:
            print(f"  Timeout on attempt {attempt}/{retries} for {indicator_code}, retrying...")
            time.sleep(3)
            if attempt == retries:
                print(f"  FAILED after {retries} attempts: {indicator_code}")
                return pd.DataFrame(columns=["country_code", "country_name", "value"])

    if len(data) < 2 or data[1] is None:
        print(f"  WARNING: {indicator_code} returned: {data[0]}")
        return pd.DataFrame(columns=["country_code", "country_name", "value"])

    rows = []
    for rec in data[1]:
        if rec["value"] is not None:
            rows.append({
                "country_code": rec["countryiso3code"],
                "country_name": rec["country"]["value"],
                "value": rec["value"],
            })
    return pd.DataFrame(rows)


agg_codes = ["ARB","EAS","EAP","ECS","ECA","EMU","EUU","HIC","IBD","IBT","IDA","IDB","IDX",
"LCN","LAC","LDC","LIC","LMC","LMY","MIC","MNA","NAC","OED","SAS","SSA","SSF","SST","WLD",
"AFE","AFW","CSS","CEB","EAR","LTE","MEA","OSS","PSS","PST","PRE","TEA","TEC","TLA","TMN",
"TSA","TSS","HPC","INX","UMC"]

merged = None
for code, name in indicators.items():
    print(f"Fetching {name} ({code}) for all countries...")
    df = fetch_indicator_all(code)
    if df.empty:
        print(f"  -> no data for {name}")
        continue
    df = df.rename(columns={"value": name})[["country_code", "country_name", name]]
    merged = df if merged is None else merged.merge(df, on=["country_code", "country_name"], how="outer")
    time.sleep(1)  # be polite to the API between calls

merged = merged[~merged["country_code"].isin(agg_codes)]
merged["price_level_ratio"] = merged["ppp_conversion_factor"] / merged["official_exchange_rate"]
merged = merged.dropna(subset=["gni_per_capita_ppp", "price_level_ratio"])
merged = merged.sort_values("country_name")

print(f"\nTotal countries with usable data: {len(merged)}")
print(merged[["country_name", "gni_per_capita_ppp", "price_level_ratio",
              "inflation_pct", "education_spend_pct_gdp"]].to_string(index=False))

merged.to_csv("all_countries_affordability.csv", index=False)
print("\nSaved -> all_countries_affordability.csv")

Fetching gni_per_capita_ppp (NY.GNP.PCAP.PP.CD) for all countries...
Fetching ppp_conversion_factor (PA.NUS.PPP) for all countries...
Fetching official_exchange_rate (PA.NUS.FCRF) for all countries...
Fetching inflation_pct (FP.CPI.TOTL.ZG) for all countries...
Fetching education_spend_pct_gdp (SE.XPD.TOTL.GD.ZS) for all countries...

Total countries with usable data: 201
                  country_name  gni_per_capita_ppp  price_level_ratio  inflation_pct  education_spend_pct_gdp
                   Afghanistan              2250.0       1.722620e-01      -6.601186                 4.343190
                       Albania             28170.0       4.293289e-01       2.146186                 3.091530
                       Algeria             18240.0       3.255632e-01       1.418352                 8.977720
                       Andorra             80950.0       6.806656e-01            NaN                 1.898680
                        Angola              9840.0       3.541320e-01      

In [ ]:
"""
Study-Abroad Affordability Comparison Tool — fully API-based
Fetches ALL countries live from the World Bank API, then scores
and compares them — no intermediate file reading.
"""

import requests
import pandas as pd
import time

# ============================================================
# STEP 1: Fetch indicators for ALL countries from the API
# ============================================================
indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",
    "PA.NUS.PPP": "ppp_conversion_factor",
    "PA.NUS.FCRF": "official_exchange_rate",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp",
}

BASE_URL = "https://api.worldbank.org/v2/country"


def fetch_indicator_all(indicator_code, retries=3):
    """Fetch the most recent non-empty value per country (mrnev=1)."""
    url = f"{BASE_URL}/all/indicator/{indicator_code}"
    params = {"format": "json", "mrnev": 1, "per_page": 400}

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            break
        except requests.exceptions.ReadTimeout:
            print(f"  Timeout on attempt {attempt}/{retries} for {indicator_code}, retrying...")
            time.sleep(3)
            if attempt == retries:
                print(f"  FAILED after {retries} attempts: {indicator_code}")
                return pd.DataFrame(columns=["country_code", "country_name", "value"])

    if len(data) < 2 or data[1] is None:
        print(f"  WARNING: {indicator_code} returned: {data[0]}")
        return pd.DataFrame(columns=["country_code", "country_name", "value"])

    rows = []
    for rec in data[1]:
        if rec["value"] is not None:
            rows.append({
                "country_code": rec["countryiso3code"],
                "country_name": rec["country"]["value"],
                "value": rec["value"],
            })
    return pd.DataFrame(rows)


agg_codes = ["ARB","EAS","EAP","ECS","ECA","EMU","EUU","HIC","IBD","IBT","IDA","IDB","IDX",
"LCN","LAC","LDC","LIC","LMC","LMY","MIC","MNA","NAC","OED","SAS","SSA","SSF","SST","WLD",
"AFE","AFW","CSS","CEB","EAR","LTE","MEA","OSS","PSS","PST","PRE","TEA","TEC","TLA","TMN",
"TSA","TSS","HPC","INX","UMC"]

df = None
for code, name in indicators.items():
    print(f"Fetching {name} ({code}) for all countries...")
    part = fetch_indicator_all(code)
    if part.empty:
        print(f"  -> no data for {name}")
        continue
    part = part.rename(columns={"value": name})[["country_code", "country_name", name]]
    df = part if df is None else df.merge(part, on=["country_code", "country_name"], how="outer")
    time.sleep(1)

df = df[~df["country_code"].isin(agg_codes)]
df["price_level_ratio"] = df["ppp_conversion_factor"] / df["official_exchange_rate"]
df = df.dropna(subset=["gni_per_capita_ppp", "price_level_ratio"])
df = df.sort_values("country_name")

print(f"\nTotal countries with usable data: {len(df)}")

# ============================================================
# STEP 2: Build the affordability score (same data, no file needed)
# ============================================================
df['affordability_score'] = (
    df['gni_per_capita_ppp'].rank(ascending=False)
    + df['price_level_ratio'].rank(ascending=True)
    + df['inflation_pct'].rank(ascending=True)
)
df = df.sort_values('affordability_score')

print("\n=== Top 10 most affordable countries overall ===")
print(df[['country_name', 'gni_per_capita_ppp', 'price_level_ratio',
          'inflation_pct', 'affordability_score']].head(10).to_string(index=False))

print("\n=== 10 least affordable countries overall ===")
print(df[['country_name', 'gni_per_capita_ppp', 'price_level_ratio',
          'inflation_pct', 'affordability_score']].tail(10).to_string(index=False))

# Still saved, as a record — but the comparisons below don't depend on re-reading it
df.to_csv('all_countries_with_score.csv', index=False)
print("\nSaved -> all_countries_with_score.csv")


# ============================================================
# STEP 3: Compare any two countries — using the data already in memory
# ============================================================
def compare(country_a, country_b, data=df):
    sub = data[data['country_name'].isin([country_a, country_b])]
    if len(sub) < 2:
        found = sub['country_name'].tolist()
        missing = [c for c in [country_a, country_b] if c not in found]
        print(f"Not found in data: {missing}")
        print("Tip: check exact spelling, e.g. 'Korea, Rep.' not 'South Korea', "
              "'Egypt, Arab Rep.' not 'Egypt', 'Russian Federation' not 'Russia'.")
        return
    print(f"\n--- {country_a} vs {country_b} ---")
    print(sub[['country_name', 'gni_per_capita_ppp', 'price_level_ratio',
               'inflation_pct', 'education_spend_pct_gdp', 'affordability_score']].to_string(index=False))


compare('Pakistan', 'United Kingdom')
compare('Pakistan', 'Malaysia')
compare('Pakistan', 'China')

Fetching gni_per_capita_ppp (NY.GNP.PCAP.PP.CD) for all countries...
Fetching ppp_conversion_factor (PA.NUS.PPP) for all countries...
Fetching official_exchange_rate (PA.NUS.FCRF) for all countries...
Fetching inflation_pct (FP.CPI.TOTL.ZG) for all countries...
Fetching education_spend_pct_gdp (SE.XPD.TOTL.GD.ZS) for all countries...

Total countries with usable data: 201

=== Top 10 most affordable countries overall ===
     country_name  gni_per_capita_ppp  price_level_ratio  inflation_pct  affordability_score
Brunei Darussalam             95420.0           0.348747      -0.302994                 79.0
         Thailand             25570.0           0.306916      -0.133634                130.0
        Sri Lanka             16750.0           0.269350      -4.763860                137.0
          Bahrain             65500.0           0.430957      -0.137709                139.0
         Malaysia             40070.0           0.316272       1.380609                143.0
 Macao SAR, China

adding ML modeling K_Means Clustering

In [ ]:
"""
Study-Abroad Affordability Comparison Tool — with K-Means clustering
"""

import requests
import pandas as pd
import time
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ============================================================
# STEP 1: Fetch indicators for ALL countries (unchanged)
# ============================================================
indicators = {
    "NY.GNP.PCAP.PP.CD": "gni_per_capita_ppp",
    "PA.NUS.PPP": "ppp_conversion_factor",
    "PA.NUS.FCRF": "official_exchange_rate",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "SE.XPD.TOTL.GD.ZS": "education_spend_pct_gdp",
}
BASE_URL = "https://api.worldbank.org/v2/country"

def fetch_indicator_all(indicator_code, retries=3):
    url = f"{BASE_URL}/all/indicator/{indicator_code}"
    params = {"format": "json", "mrnev": 1, "per_page": 400}
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            break
        except requests.exceptions.ReadTimeout:
            print(f"  Timeout on attempt {attempt}/{retries}, retrying...")
            time.sleep(3)
            if attempt == retries:
                return pd.DataFrame(columns=["country_code", "country_name", "value"])
    if len(data) < 2 or data[1] is None:
        return pd.DataFrame(columns=["country_code", "country_name", "value"])
    rows = [{"country_code": r["countryiso3code"], "country_name": r["country"]["value"], "value": r["value"]}
            for r in data[1] if r["value"] is not None]
    return pd.DataFrame(rows)

agg_codes = ["ARB","EAS","EAP","ECS","ECA","EMU","EUU","HIC","IBD","IBT","IDA","IDB","IDX",
"LCN","LAC","LDC","LIC","LMC","LMY","MIC","MNA","NAC","OED","SAS","SSA","SSF","SST","WLD",
"AFE","AFW","CSS","CEB","EAR","LTE","MEA","OSS","PSS","PST","PRE","TEA","TEC","TLA","TMN",
"TSA","TSS","HPC","INX","UMC"]

df = None
for code, name in indicators.items():
    print(f"Fetching {name}...")
    part = fetch_indicator_all(code)
    if part.empty:
        continue
    part = part.rename(columns={"value": name})[["country_code", "country_name", name]]
    df = part if df is None else df.merge(part, on=["country_code", "country_name"], how="outer")
    time.sleep(1)

df = df[~df["country_code"].isin(agg_codes)]
df["price_level_ratio"] = df["ppp_conversion_factor"] / df["official_exchange_rate"]
df = df.dropna(subset=["gni_per_capita_ppp", "price_level_ratio", "inflation_pct"])
print(f"\nCountries with complete data for clustering: {len(df)}")

# ============================================================
# STEP 2: K-Means clustering — group countries by affordability
# ============================================================
features = ["gni_per_capita_ppp", "price_level_ratio", "inflation_pct"]

scaler = StandardScaler()
X = scaler.fit_transform(df[features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# Label clusters by their average purchasing power (low -> high)
cluster_order = df.groupby("cluster")["gni_per_capita_ppp"].mean().sort_values().index
label_map = {cluster_order[0]: "Budget-Friendly", cluster_order[1]: "Mid-Range", cluster_order[2]: "Premium"}
df["affordability_group"] = df["cluster"].map(label_map)

print("\n=== How many countries landed in each group ===")
print(df["affordability_group"].value_counts())

print("\n=== Average numbers per group ===")
print(df.groupby("affordability_group")[features].mean().round(2))

df.to_csv("countries_clustered.csv", index=False)
print("\nSaved -> countries_clustered.csv")

# ============================================================
# STEP 3: Compare any two countries, including their ML-assigned group
# ============================================================
def compare(country_a, country_b, data=df):
    sub = data[data["country_name"].isin([country_a, country_b])]
    if len(sub) < 2:
        print("One or both countries not found (check exact WDI spelling).")
        return
    print(f"\n--- {country_a} vs {country_b} ---")
    print(sub[["country_name", "gni_per_capita_ppp", "price_level_ratio",
               "inflation_pct", "affordability_group"]].to_string(index=False))

compare("Pakistan", "United Kingdom")
compare("Pakistan", "Malaysia")

Fetching gni_per_capita_ppp...
Fetching ppp_conversion_factor...
Fetching official_exchange_rate...
Fetching inflation_pct...
Fetching education_spend_pct_gdp...

Countries with complete data for clustering: 191

=== How many countries landed in each group ===
affordability_group
Mid-Range          133
Premium             53
Budget-Friendly      5
Name: count, dtype: int64

=== Average numbers per group ===
                     gni_per_capita_ppp  price_level_ratio  inflation_pct
affordability_group                                                      
Budget-Friendly                12460.00               0.64         161.96
Mid-Range                      15892.18               0.41           5.00
Premium                        70179.43               0.79           3.00

Saved -> countries_clustered.csv

--- Pakistan vs United Kingdom ---
  country_name  gni_per_capita_ppp  price_level_ratio  inflation_pct affordability_group
United Kingdom             64210.0           0.891582       

In [ ]:
print(df.isna().sum())
print(df.duplicated(subset=['country_code']).sum())
print(df.dtypes)

country_code               0
country_name               0
gni_per_capita_ppp         0
ppp_conversion_factor      0
official_exchange_rate     0
inflation_pct              0
education_spend_pct_gdp    2
price_level_ratio          0
cluster                    0
affordability_group        0
dtype: int64
0
country_code                object
country_name                object
gni_per_capita_ppp         float64
ppp_conversion_factor      float64
official_exchange_rate     float64
inflation_pct              float64
education_spend_pct_gdp    float64
price_level_ratio          float64
cluster                      int32
affordability_group         object
dtype: object


In [ ]:
# 4. Wrong units — look at the extreme inflation values
print(df[df['inflation_pct'] > 40][['country_name', 'inflation_pct']].to_string(index=False))

# 5. Encoding — any non-ASCII characters in country names (not necessarily an error, just worth checking)
print(df[df['country_name'].str.contains(r'[^\x00-\x7F]', regex=True)]['country_name'].tolist())

# 6. Drift — this one needs your fetch script to have kept a "year" column per indicator.
# If you didn't keep year columns, say so honestly — that itself is a finding worth logging.
print([c for c in df.columns if 'year' in c.lower()])

      country_name  inflation_pct
         Argentina     219.883929
Iran, Islamic Rep.      42.171224
             Sudan     138.808460
       South Sudan      91.440822
     Venezuela, RB     254.948535
          Zimbabwe     104.705171
[]
[]


In [ ]:
print(df.shape)
print(df.iloc[0])

(191, 10)
country_code                    ABW
country_name                  Aruba
gni_per_capita_ppp          48900.0
ppp_conversion_factor        1.3358
official_exchange_rate         1.79
inflation_pct              4.257462
education_spend_pct_gdp    3.618558
price_level_ratio          0.746257
cluster                           1
affordability_group         Premium
Name: 4, dtype: object


In [ ]:
print(df[['gni_per_capita_ppp', 'price_level_ratio', 'inflation_pct']].describe())

       gni_per_capita_ppp  price_level_ratio  inflation_pct
count          191.000000       1.910000e+02     191.000000
mean         30866.335079       5.227131e-01       8.549865
std          29287.693898       3.381509e-01      28.018347
min           1010.000000       2.687702e-12      -6.601186
25%           7915.000000       3.407233e-01       1.427343
50%          19990.000000       4.309573e-01       3.041413
75%          48765.000000       6.322475e-01       4.739767
max         135750.000000       3.522329e+00     254.948535


In [ ]:
# You don't currently have year columns, so check indirectly:
# Try re-fetching just ONE indicator with year info kept, to see the real spread
import requests
resp = requests.get("https://api.worldbank.org/v2/country/all/indicator/NY.GNP.PCAP.PP.CD",
                     params={"format": "json", "mrnev": 1, "per_page": 400}, timeout=30)
data = resp.json()
years = [rec["date"] for rec in data[1] if rec["value"] is not None]
import pandas as pd
print(pd.Series(years).value_counts().sort_index())

2007      1
2011      2
2013      1
2015      1
2022      1
2023      1
2024     14
2025    228
Name: count, dtype: int64


In [ ]:
# Check for a few notable countries that might be excluded
check_list = ["North Korea", "Korea, Dem. People's Rep.", "Monaco", "Vatican", "Taiwan", "San Marino", "Liechtenstein"]
for name in check_list:
    match = df[df['country_name'].str.contains(name, case=False, na=False)]
    print(name, "->", "FOUND" if not match.empty else "missing",
          match['country_name'].tolist() if not match.empty else "")

North Korea -> missing 
Korea, Dem. People's Rep. -> missing 
Monaco -> missing 
Vatican -> missing 
Taiwan -> missing 
San Marino -> FOUND ['San Marino']
Liechtenstein -> missing 


In [ ]:
# Check how many of your 191 have inflation data vs not (you already found 2 missing education, but check inflation too)
print("Countries missing inflation:", df['inflation_pct'].isna().sum())
# And cross-check against the earlier full list before dropna was applied, if you still have it

Countries missing inflation: 0
